# 06 — Active Domain Adaptation (YouTube + Smart Rehearsal Buffer)

**Pipeline:**
1. Setup — mount Drive, install deps, fetch src from GitHub branch  
2. YouTube preprocessing — download 3 clips, extract 16-frame sequences  
3. Load teacher + student, extract UCF101 & YouTube embeddings  
4. Domain shift analysis — visualise the gap  
5. Active Learning acquisition — score YouTube clips by uncertainty  
6. Smart Rehearsal Buffer — prototype + hard-example selection  
7. Reptile adaptation on AL-selected YouTube clips  
8. Evaluation — before vs after, forgetting metric  
9. Plots & save everything to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
import os
os.environ["GITHUB_TOKEN"] = "xx"
os.environ["KAGGLE_USERNAME"] = "xx"
os.environ["KAGGLE_KEY"] = "xx"

In [ ]:
!bash /content/drive/MyDrive/apai/fetch_src.sh

In [ ]:
!pip install -q yt-dlp umap-learn
print("Dependencies installed.")

In [ ]:
!bash /content/drive/MyDrive/apai/setup_colab.sh

In [ ]:
%run /content/src/imports.py

In [ ]:
preprocess_dataset(splits=[], target_classes=cfg.SELECTED_CLASSES)

In [ ]:
old_classes = cfg.SELECTED_CLASSES
num_classes = len(old_classes)
c2i         = {c: i for i, c in enumerate(old_classes)}

pin_memory   = device.type == "cuda"

train_ds = UCF101Clips(f"{cfg.BASE_ROOT}/train", class_to_idx=c2i)
val_ds   = UCF101Clips(f"{cfg.BASE_ROOT}/val",   class_to_idx=c2i)
test_ds  = UCF101Clips(f"{cfg.BASE_ROOT}/test",  class_to_idx=c2i)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=cfg.NUM_WORKERS, pin_memory=pin_memory)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=pin_memory)
test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=pin_memory)
print(f"Classes ({num_classes}): {old_classes}")
print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## 6 — Load Teacher & Student Models

In [ ]:
teacher = ResNet50LSTMTeacher(cfg.TEACHER_HIDDEN, num_classes, 0.4).to(device)
student = MobileNetV3SmallLSTMStudent(num_classes, cfg.STUDENT_HIDDEN, cfg.STUDENT_DROPOUT).to(device)

# ── Load teacher checkpoint ───────────────────────────────────────────────
raw_ckpt = torch.load(cfg.RESNET50_PATH, map_location=device)
mapped   = remap_teacher_checkpoint(raw_ckpt)
missing, _ = teacher.load_state_dict(mapped, strict=False)
real_missing = [k for k in missing if "num_batches_tracked" not in k]
if real_missing:
    raise RuntimeError(f"Missing keys: {real_missing[:5]}")
print(f"Teacher loaded ({len(mapped)} keys).")

# ── Load student checkpoint from Drive ────────────────────────────────────
student_ckpt = f"{cfg.CKPT_DIR}/student_ucf101.pt"
if os.path.exists(student_ckpt):
    student.load_state_dict(torch.load(student_ckpt, map_location=device))
    print(f"Student loaded from {student_ckpt}")
else:
    print("WARNING: No student checkpoint found at:", student_ckpt)
    print("  -> Run notebook 3.0_reptile_kd.ipynb first to produce a trained student.")

for p in teacher.parameters():
    p.requires_grad_(False)
teacher.eval()
print(f"Teacher frozen. Student params: {sum(p.numel() for p in student.parameters()):,}")

## 7 — Extract UCF101 Embeddings (cached to Drive)

In [ ]:
UCF_EMB_CACHE = f"{cfg.DRIVE_PROJECT}/cache/ucf101_embeddings.pt"
os.makedirs(os.path.dirname(UCF_EMB_CACHE), exist_ok=True)

if os.path.exists(UCF_EMB_CACHE):
    print("Loading cached UCF101 embeddings from Drive...")
    cache = torch.load(UCF_EMB_CACHE, map_location="cpu")
    train_s, train_t, train_y = cache["train_s"], cache["train_t"], cache["train_y"]
    val_s,   val_t,   val_y   = cache["val_s"],   cache["val_t"],   cache["val_y"]
    test_s,  test_t,  test_y  = cache["test_s"],  cache["test_t"],  cache["test_y"]
else:
    print("Extracting embeddings (first time, ~5 min)...")
    train_t, train_y = extract_embeddings(teacher, train_loader, device)
    val_t,   val_y   = extract_embeddings(teacher, val_loader,   device)
    test_t,  test_y  = extract_embeddings(teacher, test_loader,  device)
    train_s, _       = extract_embeddings(student, train_loader, device)
    val_s,   _       = extract_embeddings(student, val_loader,   device)
    test_s,  _       = extract_embeddings(student, test_loader,  device)
    torch.save({"train_s": train_s, "train_t": train_t, "train_y": train_y,
                "val_s": val_s, "val_t": val_t, "val_y": val_y,
                "test_s": test_s, "test_t": test_t, "test_y": test_y}, UCF_EMB_CACHE)
    print(f"Cached -> {UCF_EMB_CACHE}")

print(f"Train: {train_s.shape} | Val: {val_s.shape} | Test: {test_s.shape}")

## 8 — Download YouTube Videos
3 videos defined in Cell 0.

In [ ]:
for cls_name, url in cfg.YOUTUBE_CLIPS.items():
    out_path = f"{cfg.YT_RAW_DIR}/{cls_name}.mp4"
    if os.path.exists(out_path):
        print(f"  [skip] {cls_name}.mp4 already exists")
        continue
    print(f"  Downloading: {cls_name} ...")
    res = subprocess.run([
        "yt-dlp",
        "--format", "bestvideo[ext=mp4][height<=480]+bestaudio[ext=m4a]/best[ext=mp4]",
        "--merge-output-format", "mp4",
        "--output", out_path,
        "--quiet", "--no-warnings", url
    ], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"  [!] Failed: {res.stderr[:150]}")
    else:
        size_mb = os.path.getsize(out_path) / 1e6
        print(f"  [OK] {cls_name}.mp4  ({size_mb:.1f} MB)")

print("Downloads complete.")

## 9 — Preprocess YouTube Clips
Extract 16-frame clips with **identical** transforms to UCF101.

In [ ]:
yt_clips = {}   # class_name -> list of (T, C, H, W) tensors
for cls_name in cfg.YOUTUBE_CLIPS:
    if cls_name not in c2i:
        print(f"  [!] {cls_name} not in UCF101 class list — skipping")
        continue
    vpath  = f"{cfg.YT_RAW_DIR}/{cls_name}.mp4"
    clips  = extract_clips(vpath, cfg.CLIP_LEN, cfg.CLIPS_PER_VID)
    yt_clips[cls_name] = clips
    print(f"  [OK] {cls_name}: {len(clips)} clips  shape={clips[0].shape if clips else None}")

print(f"Total YouTube clips: {sum(len(v) for v in yt_clips.values())}")

## 10 — Extract YouTube Embeddings

In [ ]:
teacher.eval(); student.eval()
yt_s_dict = {}  # class -> student embs
yt_t_dict = {}  # class -> teacher embs

with torch.no_grad():
    for cls_name, clips in yt_clips.items():
        if not clips:
            continue
        batch = torch.stack(clips).to(device)  # (N, T, C, H, W)
        _, t_emb = teacher(batch)
        _, s_emb = student(batch)
        yt_s_dict[cls_name] = s_emb.cpu()
        yt_t_dict[cls_name] = t_emb.cpu()
        print(f"  {cls_name}: student {s_emb.shape} | teacher {t_emb.shape}")

# Flatten
yt_s_all = torch.cat(list(yt_s_dict.values()), dim=0)
yt_t_all = torch.cat(list(yt_t_dict.values()), dim=0)
yt_y_all = torch.cat([
    torch.full((yt_s_dict[c].shape[0],), c2i[c], dtype=torch.long)
    for c in yt_s_dict
])
print(f"YouTube embeddings: {yt_s_all.shape}")

## 11 — Domain Shift Analysis

In [ ]:
# ── Compute per-class statistics using src.active.domain_shift ──────────────
yt_class_idxs = [c2i[c] for c in yt_s_dict]
mask_ucf      = torch.isin(test_y, torch.tensor(yt_class_idxs))
ucf_s_sub     = test_s[mask_ucf]
ucf_y_sub     = test_y[mask_ucf]

ucf_stats = compute_class_stats(ucf_s_sub, ucf_y_sub)
yt_stats  = compute_class_stats(yt_s_all,  yt_y_all)
shifts    = measure_shift(ucf_stats, yt_stats, class_names=old_classes)

# ── t-SNE visualisation ───────────────────────────────────────────────────
all_embs = torch.cat([ucf_s_sub, yt_s_all]).numpy()
domains  = ["UCF101"] * ucf_s_sub.shape[0] + ["YouTube"] * yt_s_all.shape[0]
proj     = TSNE(n_components=2, perplexity=min(30, len(all_embs) - 1),
               random_state=42).fit_transform(PCA(50).fit_transform(all_embs))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for cls_name in yt_s_dict:
    idx  = c2i[cls_name]
    all_y = np.concatenate([ucf_y_sub.numpy(), yt_y_all.numpy()])
    mask  = all_y == idx
    axes[0].scatter(proj[mask, 0], proj[mask, 1], label=cls_name, alpha=0.6, s=20)
axes[0].set_title("by class"); axes[0].legend(fontsize=7); axes[0].axis("off")
for dom, col in [("UCF101", "steelblue"), ("YouTube", "tomato")]:
    m = np.array(domains) == dom
    axes[1].scatter(proj[m, 0], proj[m, 1], label=dom, alpha=0.6, s=20, c=col)
axes[1].set_title("UCF101 vs YouTube"); axes[1].legend(); axes[1].axis("off")
plt.suptitle("t-SNE: Domain Shift", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/domain_shift_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR}/domain_shift_tsne.png")


## 12 — Active Learning Acquisition
Score YouTube clips by uncertainty, select top-K per class.

In [ ]:
# Load the trained head for scoring YouTube clips
head_for_scoring = EmbeddingHead(
    cfg.STUDENT_HIDDEN, cfg.TEACHER_HIDDEN,
    num_classes=num_classes,
    dropout_p=cfg.HEAD_DROPOUT,
).to(device)

head_ckpt = f"{cfg.CKPT_DIR}/head_ucf101.pt"
if os.path.exists(head_ckpt):
    head_for_scoring.load_state_dict(torch.load(head_ckpt, map_location=device))
    print(f"Loaded trained head from {head_ckpt}")
else:
    print("No saved head found — using random init head.")
    print("Run notebook 05_kd_reptile.ipynb first for a trained head.")

# ── Score & select using src.active.acquisition.select_top_k ─────────────
al_selected = select_top_k(
    embs_by_class=yt_s_dict,
    strategy=cfg.AL_STRATEGY,
    k=cfg.AL_BUDGET_PER_CLASS,
    head=head_for_scoring,
    device=device,
)

# Build AL tensors from selected indices
al_s = torch.cat([yt_s_dict[c][al_selected[c]] for c in al_selected])
al_t = torch.cat([yt_t_dict[c][al_selected[c]] for c in al_selected])
al_y = torch.cat([
    torch.full((len(al_selected[c]),), c2i[c], dtype=torch.long)
    for c in al_selected
])
print(f"\nAL total: {al_s.shape[0]} clips  strategy={AL_STRATEGY}")


## 13 — Smart Rehearsal Buffer
Prototype-aware: stores UCF101 hard exemplars + AL-selected YouTube clips.

In [ ]:
# SmartReplayBuffer is imported from src.cl.smart_replay — no inline redefinition needed.
buffer = SmartReplayBuffer(max_per_class=10)

print("Adding UCF101 exemplars (hard selection)...")
for cls_name in old_classes:
    idx  = c2i[cls_name]
    mask = train_y == idx
    if mask.sum() > 0:
        buffer.add_class(idx, train_s[mask], train_t[mask], strategy="hard")

print("Adding AL-selected YouTube clips (diverse selection)...")
for cls_name in al_selected:
    idx   = c2i[cls_name]
    s_yt  = yt_s_dict[cls_name][al_selected[cls_name]]
    t_yt  = yt_t_dict[cls_name][al_selected[cls_name]]
    if idx in buffer.s_store:
        s_combined = torch.cat([buffer.s_store[idx], s_yt])
        t_combined = torch.cat([buffer.t_store[idx], t_yt])
    else:
        s_combined, t_combined = s_yt, t_yt
    buffer.add_class(idx, s_combined, t_combined, strategy="diverse")

buffer.summary(class_names=old_classes)
buf_s, buf_t, buf_y = buffer.get_all()
print(f"Total buffer: {buf_s.shape[0]} samples")


## 14 — Baseline Accuracy (before adaptation)

In [ ]:
head = EmbeddingHead(cfg.STUDENT_HIDDEN, cfg.TEACHER_HIDDEN,
                     num_classes=num_classes, dropout_p=cfg.HEAD_DROPOUT).to(device)
if os.path.exists(head_ckpt):
    head.load_state_dict(torch.load(head_ckpt, map_location=device))
    print(f"Loaded head from {head_ckpt}")

ucf_acc_before = eval_head(head, test_s,   test_y,   device)
yt_acc_before  = eval_head(head, yt_s_all, yt_y_all, device)

print(f"UCF101 accuracy  (before): {ucf_acc_before:.2%}")
print(f"YouTube accuracy (before): {yt_acc_before:.2%}")
print(f"Domain gap               : {ucf_acc_before - yt_acc_before:.2%}")

## 15 — Reptile Adaptation on AL-Selected YouTube Clips

In [ ]:
head_adapted = copy.deepcopy(head)

# ── Reptile adaptation using src.active.domain_shift.reptile_adapt ──────────
head_adapted = reptile_adapt(
    head          = head_adapted,
    al_s          = al_s,
    al_t          = al_t,
    al_y          = al_y,
    buffer        = buffer,
    device        = device,
    episodes      = cfg.ADAPT_EPISODES,
    inner_lr      = cfg.ADAPT_INNER_LR,
    inner_steps   = cfg.ADAPT_INNER_STEPS,
    epsilon       = cfg.ADAPT_EPSILON,
    mse_weight    = 0.5,
    label_smoothing = 0.1,
    eval_s        = yt_s_all,
    eval_y        = yt_y_all,
    log_every     = 10,
)
print("Adaptation complete.")


## 16 — Evaluation: Before vs After

In [ ]:
ucf_acc_after = eval_head(head_adapted, test_s,   test_y,   device)
yt_acc_after  = eval_head(head_adapted, yt_s_all, yt_y_all, device)
forgetting    = ucf_acc_before - ucf_acc_after
yt_gain       = yt_acc_after   - yt_acc_before

W = 58
print("=" * W)
print(f"  {'Metric':<35} {'Before':>8} {'After':>8} {'Delta':>8}")
print("-" * W)
print(f"  {'UCF101 accuracy':<35} {ucf_acc_before:>7.2%} "
      f"{ucf_acc_after:>7.2%}  {-forgetting:>+.2%}")
print(f"  {'YouTube accuracy':<35} {yt_acc_before:>7.2%} "
      f"{yt_acc_after:>7.2%}  {yt_gain:>+.2%}")
print("=" * W)

print("\nPer-class YouTube:")
for cls_name in yt_s_dict:
    idx   = c2i[cls_name]
    mask  = yt_y_all == idx
    s_c   = yt_s_all[mask]; y_c = yt_y_all[mask]
    acc_b = eval_head(head,         s_c, y_c, device)
    acc_a = eval_head(head_adapted, s_c, y_c, device)
    print(f"  {cls_name:<20}: {acc_b:.2%} -> {acc_a:.2%}  ({acc_a-acc_b:+.2%})")

results = dict(ucf_acc_before=ucf_acc_before, ucf_acc_after=ucf_acc_after,
               yt_acc_before=yt_acc_before,   yt_acc_after=yt_acc_after,
               forgetting=forgetting, yt_gain=yt_gain,
               al_strategy=cfg.AL_STRATEGY, al_budget=AL_BUDGET_PER_CLASS)

## 17 — Plots

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Plot 1: Before vs After
ax1 = fig.add_subplot(gs[0, 0])
labels_bar = ["UCF101", "YouTube"]
bv = [ucf_acc_before*100, yt_acc_before*100]
av = [ucf_acc_after*100,  yt_acc_after*100]
x  = np.arange(2); w = 0.35
ax1.bar(x-w/2, bv, w, label="Before", color="steelblue",  alpha=0.8)
ax1.bar(x+w/2, av, w, label="After",  color="darkorange", alpha=0.8)
for i, (b, a) in enumerate(zip(bv, av)):
    ax1.text(i-w/2, b+1, f"{b:.1f}%", ha="center", fontsize=8)
    ax1.text(i+w/2, a+1, f"{a:.1f}%", ha="center", fontsize=8)
ax1.set_xticks(x); ax1.set_xticklabels(labels_bar)
ax1.set_ylim(0, 110); ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Before vs After Adaptation"); ax1.legend()

# Plot 2: Gain vs Forgetting
ax2 = fig.add_subplot(gs[0, 1])
deltas = [yt_gain*100, -forgetting*100]
cols   = ["green" if d >= 0 else "red" for d in deltas]
ax2.bar(["YouTube\nGain", "UCF101\nRetention (+= good)"], deltas,
        color=cols, alpha=0.8)
ax2.axhline(0, color="k", lw=0.8, ls="--")
ax2.set_ylabel("Accuracy change (pp)")
ax2.set_title(f"Gain vs Forgetting\nAL={cfg.AL_STRATEGY}, K={cfg.AL_BUDGET_PER_CLASS}")
for i, d in enumerate(deltas):
    ax2.text(i, d+0.3*np.sign(d+0.001), f"{d:+.1f}pp", ha="center", fontsize=9)

# Plot 3: Per-class YouTube
ax3 = fig.add_subplot(gs[0, 2])
cls_names_p, acc_b_p, acc_a_p = [], [], []
for cls_name in yt_s_dict:
    idx  = c2i[cls_name]
    mask = yt_y_all == idx
    sc   = yt_s_all[mask]; yc = yt_y_all[mask]
    cls_names_p.append(cls_name)
    acc_b_p.append(eval_head(head,         sc, yc, device)*100)
    acc_a_p.append(eval_head(head_adapted, sc, yc, device)*100)
x3 = np.arange(len(cls_names_p))
ax3.bar(x3-w/2, acc_b_p, w, label="Before", color="steelblue",  alpha=0.8)
ax3.bar(x3+w/2, acc_a_p, w, label="After",  color="darkorange", alpha=0.8)
ax3.set_xticks(x3); ax3.set_xticklabels(cls_names_p, rotation=15, ha="right")
ax3.set_ylim(0, 110); ax3.set_title("Per-Class YouTube"); ax3.legend()

# Plot 4: t-SNE embedding space
ax4 = fig.add_subplot(gs[1, :2])
all_e   = torch.cat([ucf_s_sub, yt_s_all]).numpy()
doms    = ["UCF101"]*ucf_s_sub.shape[0] + ["YouTube"]*yt_s_all.shape[0]
proj_p  = TSNE(n_components=2, perplexity=min(30, len(all_e)-1),
               random_state=42).fit_transform(PCA(50).fit_transform(all_e))
for dom, col in [("UCF101","steelblue"),("YouTube","tomato")]:
    m = np.array(doms) == dom
    ax4.scatter(proj_p[m,0], proj_p[m,1], label=dom, alpha=0.6, s=25, c=col)
ax4.set_title("Embedding Space: UCF101 vs YouTube")
ax4.legend(); ax4.axis("off")

# Plot 5: Buffer composition
ax5 = fig.add_subplot(gs[1, 2])
buf_counts = {old_classes[k]: v.shape[0] for k, v in buffer.s_store.items()}
ax5.barh(list(buf_counts.keys()), list(buf_counts.values()),
         color="mediumseagreen", alpha=0.8)
ax5.set_xlabel("Samples in buffer")
ax5.set_title("Smart Replay Buffer Composition")

plt.suptitle(
    f"Active Domain Adaptation | AL={cfg.cfg.cfg.AL_STRATEGY.upper()} K={cfg.AL_BUDGET_PER_CLASS}\n"
    f"YouTube gain: {yt_gain*100:+.1f}pp | Forgetting: {forgetting*100:+.1f}pp",
    fontsize=12, fontweight="bold")
plt.savefig(f"{cfg.RESULTS_DIR}/adaptation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR}/adaptation_results.png")

## 18 — Save Everything to Drive

In [ ]:
torch.save(head_adapted.state_dict(), f"{cfg.CKPT_DIR}/head_adapted_yt.pt")
print(f"[OK] Adapted head -> {cfg.CKPT_DIR}/head_adapted_yt.pt")

torch.save({"s_store": buffer.s_store, "t_store": buffer.t_store,
            "mu_store": buffer.mu_store}, f"{cfg.CKPT_DIR}/smart_replay_buffer.pt")
print(f"[OK] Buffer -> {cfg.CKPT_DIR}/smart_replay_buffer.pt")

with open(f"{cfg.RESULTS_DIR}/results.json", "w") as f:
    json.dump({k: float(v) if isinstance(v, (float, int)) else v
               for k, v in results.items()}, f, indent=2)
print(f"[OK] Results JSON -> {cfg.RESULTS_DIR}/results.json")

with open(f"{cfg.RESULTS_DIR}/al_selections.json", "w") as f:
    json.dump({k: [int(i) for i in v] for k, v in al_selected.items()}, f, indent=2)
print(f"[OK] AL selections -> {cfg.RESULTS_DIR}/al_selections.json")

print("\n=== All outputs saved to Drive ===")
print(f"  checkpoints : {cfg.CKPT_DIR}/")
print(f"  results     : {cfg.RESULTS_DIR}/")